# 変更メモ（派生ノートブック）

- 対象: `[1]dpc-byt5-retrain-tokenizer-v3.ipynb`
- 元: `[1]dpc-byt5-retrain-tokenizer-v2.ipynb`
- 種別: バージョン更新
- 変更点:
  - 学習データを curated 版 `train.curated.v002.xlsx` に差し替え（`oare_id`, `transliteration_x`, `translation` を使用）
  - `.codex/docs/normalization_replacements.md` の観点を反映（`<gap>` 統一・縮約、下付き数字、長い小数の短縮など）
  - `<gap>. <gap>` / `<gap>, <gap>` のように句読点を挟む連続 `<gap>` も `<gap>` に縮約
- 変更理由/仮説: curated train + `<gap>` 正規化で、tokenizer 学習用コーパスと翻訳学習のノイズを減らしたい

## 使い分け
- まずは `Config.MODEL_PRESET = "p100-medium"` のまま実行
- OOMする場合: `Config.MAX_LENGTH` を 512→384/256、または `Config.BATCH_SIZE` を 1 のまま `Config.GRAD_ACCUMULATION_STEPS` を増やす
- さらに攻める場合: `Config.MODEL_PRESET = "t5-base-p100"`（遅い/OOMしやすいので注意）


# Deep Past Initiative – Machine Translation

## Takeaway

- When the first letter of a word is capitalized it implies the word is a personal name or a place name (i.e. proper noun). When the word is in ALL CAPS, that implies it is a Sumerian logogram and was written in place of the Akkadian syllabic spelling for scribal simplicity.

- Can use the published_text's transliteration as the cleaned version for training.

- Can use the normed words in Lexicon to swap for training.

## Facts

- The transliteration words len < 150, translation words len < 300


In [1]:
!pip install -q evaluate sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.5 MB/s eta 0:00:00


In [2]:
import os 
import gc
import re
import unicodedata
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)
from sentence_transformers import SentenceTransformer, util
import evaluate
import matplotlib.pyplot as plt
import sentencepiece as spm
from transformers import T5Tokenizer

2026-03-13 16:25:57.710607: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773419157.905581      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773419157.962850      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773419158.431791      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773419158.431827      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773419158.431830      23 computation_placer.cc:177] computation placer alr

In [3]:
class Config:
    # SentencePieceで学習したトークナイザ + T5をスクラッチ学習する実験
    # - 事前学習済みの ByT5 重みは使わない（このノートブックでは tokenizer も作り直す）
    # - `MODEL_NAME` は「この実験の識別子」としてのみ使う

    # ===== Model size preset =====
    # P100(16GB)で回しやすい順（目安）:
    # - p100-medium（推奨）: d_model=512, layers=12
    # - t5-base-p100        : d_model=768, layers=12（遅い/OOMしやすい）
    MODEL_PRESET = "p100-medium"

    MODEL_NAME = f"t5-scratch-spm-{MODEL_PRESET}"

    # SentencePiece + スクラッチ学習のため、長さはまず 512 で打ち切る
    MAX_LENGTH = 512

    # ===== Training (P100想定) =====
    # 大型化に伴い、per-device batch を小さくして grad accum で稼ぐ
    BATCH_SIZE = 1
    GRAD_ACCUMULATION_STEPS = 8

    EPOCHS = 100
    LEARNING_RATE = 2e-4

    OUTPUT_DIR = f"./{MODEL_NAME}-akkadian-model"

    PATIENCE = 10

    DRY_RUN = False


In [4]:
def seed_everything(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)
    
seed_everything()

In [5]:
INPUT_DIR = "/kaggle/input/competitions/deep-past-initiative-machine-translation"

# ------------------------------------------
# Train: curated xlsx を優先して読む
# ------------------------------------------
# Kaggle 上で使う場合は、curated xlsx を Dataset として追加して、
# `/kaggle/input/.../train.curated.v002.xlsx` が参照できるようにしてください。
USE_CURATED_TRAIN = True
CURATED_TRAIN_XLSX_CANDIDATES = [
    "/kaggle/input/datasets/tatsuokoshida/dpc-curated-train/train.curated.v002.xlsx",
]


def load_base_train_df():
    if USE_CURATED_TRAIN:
        curated_path = None
        for p in CURATED_TRAIN_XLSX_CANDIDATES:
            if os.path.exists(p):
                curated_path = p
                break
        if curated_path is None:
            raise FileNotFoundError(
                "curated train xlsx が見つかりません。CURATED_TRAIN_XLSX_CANDIDATES を確認してください。"
            )

        df = pd.read_excel(curated_path)
        df = df.rename(columns={"transliteration_x": "transliteration"})

        required_cols = {"oare_id", "transliteration", "translation"}
        missing_cols = required_cols - set(df.columns)
        assert not missing_cols, f"curated xlsx の必須カラムが不足しています: {sorted(missing_cols)}"

        df = df[["oare_id", "transliteration", "translation"]].copy()
        df["transliteration"] = df["transliteration"].fillna("").astype(str)
        df["translation"] = df["translation"].fillna("").astype(str)

        df.attrs["train_source"] = curated_path
        return df

    df = pd.read_csv(f"{INPUT_DIR}/train.csv")
    df.attrs["train_source"] = f"{INPUT_DIR}/train.csv"
    return df


base_train_df = load_base_train_df()
test_df = pd.read_csv(f"{INPUT_DIR}/test.csv")

train_sent_df = pd.read_csv("/kaggle/input/notebooks/zhangyue199/dpc-sentence-alignment-oare-firstword/sentence_alignment.csv")
train_sent_df.rename(columns={
    "translit_sent": "transliteration",
    "translation_sent_train": "translation"
}, inplace=True)

train_df = pd.concat([base_train_df, train_sent_df[base_train_df.columns]], ignore_index=True)

train_source = base_train_df.attrs.get("train_source", "(unknown)")
print(f"Train Data: {len(train_df)} docs. | base_source={train_source} | +sentence_alignment={len(train_sent_df)}")

Train Data: 2616 docs. | base_source=/kaggle/input/datasets/tatsuokoshida/dpc-curated-train/train.curated.v002.xlsx | +sentence_alignment=1052


# Build Corpus

In [6]:
SRC_COL = "transliteration"
TGT_COL = "translation"

# Tokenizer 学習用コーパス: `.codex/docs/normalization_replacements.md` の観点を入れて正規化
# - `<gap>` の表記揺れを統一し、連続 `<gap>`（`<gap>. <gap>`, `<gap>, <gap>` を含む）を縮約
# - 下付き数字、長い小数、ダッシュ類などの揺れを軽く寄せる

_SUBSCRIPT_NUM_MAP = str.maketrans({
    "₀": "0", "₁": "1", "₂": "2", "₃": "3", "₄": "4",
    "₅": "5", "₆": "6", "₇": "7", "₈": "8", "₉": "9",
})

_TRANSLATION_D1_PATTERNS = [
    (re.compile(r"\b(?:fem|sing|pl|plural)\.?(?=\s|$)", flags=re.IGNORECASE), " "),
    (re.compile(r"\(\?\)"), " "),
]

_GAP_MARKER_PATTERNS = [
    (re.compile(r"\(\s*(?:large\s+)?break\s*\)", flags=re.IGNORECASE), " <gap> "),
    (re.compile(r"\(\s*\d+\s+broken\s+lines?\s*\)", flags=re.IGNORECASE), " <gap> "),
    (re.compile(r"<\s*big_gap\s*>", flags=re.IGNORECASE), " <gap> "),
    (re.compile(r"\bbig_gap\b", flags=re.IGNORECASE), " <gap> "),
    (re.compile(r"\[\s*[xX]+\s*\]"), " <gap> "),
    (re.compile(r"\b[xX]{1,}\b"), " <gap> "),
]


def _collapse_gaps(text: str) -> str:
    text = re.sub(r"\s*<gap>\s*", " <gap> ", text)
    while True:
        new_text = re.sub(r"<gap>\s*[\.,]\s*<gap>", "<gap>", text)
        new_text = re.sub(r"<gap>\s+<gap>", "<gap>", new_text)
        if new_text == text:
            break
        text = new_text
    return text


def normalize_transliteration(text):
    text = "" if text is None else str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.translate(_SUBSCRIPT_NUM_MAP)
    text = re.sub(r"[‐‑–—]", "-", text)
    text = text.replace("ₓ", "")
    text = re.sub(r"(\d+\.\d{4})\d+", r"\1", text)

    text = text.replace("…", " <gap> ")
    for pat, repl in _GAP_MARKER_PATTERNS:
        text = pat.sub(repl, text)
    text = _collapse_gaps(text)

    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_translation_d1(text):
    text = "" if text is None else str(text)
    text = unicodedata.normalize("NFKC", text)
    for pattern, repl in _TRANSLATION_D1_PATTERNS:
        text = pattern.sub(repl, text)
    text = re.sub(r"\bPN\b", " <gap> ", text)
    text = _collapse_gaps(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def postprocess_translation(text):
    return normalize_translation_d1(text)


def clean_transliteration(s):
    return normalize_transliteration(s)


def clean_translation(s):
    return normalize_translation_d1(s)

with open("spm_corpus.txt", "w", encoding="utf-8") as f:
    for _, row in train_df.iterrows():
        f.write(clean_transliteration(row[SRC_COL]) + "\n")
        f.write(clean_translation(row[TGT_COL]) + "\n")

# Sentencepiece Tokenizer

In [7]:
spm.SentencePieceTrainer.train(
    input="/kaggle/working/spm_corpus.txt",
    model_prefix="akkadian_spm",
    vocab_size=3000,
    model_type="unigram",         # better for low-resource
    character_coverage=1.0,
    user_defined_symbols=[
        "<DIV>", "<NUM>"
    ],
    bos_id=-1,
    eos_id=1,
    pad_id=0,
    unk_id=2
)

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: /kaggle/working/spm_corpus.txt
  input_format: 
  model_prefix: akkadian_spm
  model_type: UNIGRAM
  vocab_size: 3000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: <DIV>
  user_defined_symbols: <NUM>
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 2
  bos_id: -1
  eos_id: 1
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
 

# Load Tokenizer

In [8]:
tokenizer = T5Tokenizer(
    vocab_file="/kaggle/working/akkadian_spm.model",
    extra_ids=0,
    pad_token="<pad>",
    eos_token="</s>",
    unk_token="<unk>"
)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


# T5 model for training

In [9]:
from transformers import T5Config, T5ForConditionalGeneration

MODEL_PRESETS = {
    # 元ノート（小型）
    "original-small": {
        "d_model": 256,
        "d_ff": 1024,
        "num_layers": 6,
        "num_decoder_layers": 6,
        "num_heads": 8,
    },
    # P100で回しやすい範囲での大型化（推奨）
    "p100-medium": {
        "d_model": 512,
        "d_ff": 2048,
        "num_layers": 12,
        "num_decoder_layers": 12,
        "num_heads": 8,
    },
    # さらに大型（t5-base相当）。OOMしたら MAX_LENGTH を下げる
    "t5-base-p100": {
        "d_model": 768,
        "d_ff": 3072,
        "num_layers": 12,
        "num_decoder_layers": 12,
        "num_heads": 12,
    },
}

if Config.MODEL_PRESET not in MODEL_PRESETS:
    raise ValueError(f"Unknown MODEL_PRESET: {Config.MODEL_PRESET}. choices={list(MODEL_PRESETS)}")

preset = MODEL_PRESETS[Config.MODEL_PRESET]

config = T5Config(
    vocab_size=tokenizer.vocab_size,
    dropout_rate=0.1,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
    decoder_start_token_id=tokenizer.pad_token_id,
    **preset,
)

model = T5ForConditionalGeneration(config)

# 大型モデルはメモリが厳しいのでチェックポイント化
model.gradient_checkpointing_enable()
model.config.use_cache = False


# Create Batched Dataset

In [10]:
from datasets import Dataset

def preprocess(examples):
    prefix = "translate Akkadian to English: "

    src_norm = [clean_transliteration(ex) for ex in examples[SRC_COL]]
    tgt_norm = [clean_translation(ex) for ex in examples[TGT_COL]]

    inputs = [prefix + ex for ex in src_norm]
    targets = [ex for ex in tgt_norm]

    model_inputs = tokenizer(
        inputs,
        max_length=Config.MAX_LENGTH,
        truncation=True,
        text_target=targets
    )
    return model_inputs
    
dataset = Dataset.from_pandas(train_df)
split_dataset = dataset.train_test_split(test_size=0.15, seed=42)

train_dataset = split_dataset["train"].map(preprocess, batched=True)
val_dataset = split_dataset["test"].map(preprocess, batched=True)

Map:   0%|          | 0/2223 [00:00<?, ? examples/s]

Map:   0%|          | 0/393 [00:00<?, ? examples/s]

# Training

In [11]:
from sacrebleu.metrics import CHRF
import numpy as np

# update the metrics as competition metrics collapse during training phase
chrf = CHRF(word_order=2)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_preds = [postprocess_translation(p) for p in decoded_preds]

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_labels = [postprocess_translation(l) for l in decoded_labels]

    # sentence-level chrF (macro)
    sent_scores = [
        chrf.sentence_score(p.strip(), [l.strip()]).score
        for p, l in zip(decoded_preds, decoded_labels)
    ]

    return {
        "sentence_chrf": float(np.mean(sent_scores))
    }


In [12]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback, GenerationConfig

# GenerationConfig(
#     max_length=512,
#     num_beams=4,
#     no_repeat_ngram_size=4,
#     repetition_penalty=1.3,
#     length_penalty=1.1,
#     early_stopping=True,
# )

EPOCH = 3 if Config.DRY_RUN else Config.EPOCHS

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir=Config.OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=Config.LEARNING_RATE,      # higher LR since training from scratch
    per_device_train_batch_size=Config.BATCH_SIZE,
    per_device_eval_batch_size=Config.BATCH_SIZE,
    gradient_accumulation_steps=Config.GRAD_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    optim="adafactor",
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=EPOCH,
    fp16=True,
    logging_steps=100,
    report_to="none",
    # ===== EARLY STOPPING CONFIG =====
    metric_for_best_model="eval_loss",
    load_best_model_at_end=True,
    greater_is_better=False,
    # predict_with_generate=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    # early stopping
    # compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=Config.PATIENCE)],

)

trainer.train()


/tmp/ipykernel_23/917979356.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Epoch,Training Loss,Validation Loss
1,5.338900,4.783621
2,4.336100,4.208674
3,3.985600,3.980287
4,3.694200,3.792223
5,3.519900,3.713481
6,3.254600,3.585518
7,3.127000,3.520115
8,3.021200,3.452187
9,2.837300,3.406531
10,2.733500,3.395403


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=11676, training_loss=2.1298451353401795, metrics={'train_runtime': 15678.889, 'train_samples_per_second': 14.178, 'train_steps_per_second': 1.773, 'total_flos': 8667853310398464.0, 'train_loss': 2.1298451353401795, 'epoch': 42.0})

# Save

In [13]:
trainer.save_model(Config.OUTPUT_DIR)
tokenizer.save_pretrained(Config.OUTPUT_DIR)
print(f"Model saved to {Config.OUTPUT_DIR}")

Model saved to ./t5-scratch-spm-p100-medium-akkadian-model


In [14]:
# predictions = trainer.predict(val_dataset)
# labels = predictions.label_ids
# labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
# decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

# decoded_labels = [[label.strip()] for label in decoded_labels]
# pred_ids = predictions.predictions

# if isinstance(pred_ids, tuple):
#     pred_ids = pred_ids[0]

# decoded_preds = tokenizer.batch_decode(
#     pred_ids,
#     skip_special_tokens=True,
#     clean_up_tokenization_spaces=True,
# )
# decoded_preds = [pred.strip() for pred in decoded_preds]

# Dry run Inference

In [15]:
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

MAX_LENGTH = 512
DEVICE = "cuda"

tokenizer_loaded = AutoTokenizer.from_pretrained(Config.OUTPUT_DIR)
model_loaded = AutoModelForSeq2SeqLM.from_pretrained(Config.OUTPUT_DIR).to("cuda")
model_loaded.eval()


class InferenceDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts = df['transliteration'].astype(str).tolist()
        self.texts = [normalize_transliteration(i) for i in self.texts]
        self.texts = ["translate Akkadian to English: " + i for i in self.texts]
        self.translation = df['translation'].astype(str).tolist()
        self.translation = [normalize_translation_d1(i) for i in self.translation]
        self.tokenizer = tokenizer
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        translation = self.translation[idx]
        inputs = self.tokenizer(
            text, 
            max_length=MAX_LENGTH, 
            padding="max_length", 
            truncation=True, 
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "translation": translation
        }

test_df = train_df.sample(5)
test_dataset = InferenceDataset(test_df, tokenizer_loaded)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)


print("Starting Inference...")
all_predictions = []
actuals = []
with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        actuals.extend(batch['translation'])
  
        outputs = model_loaded.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=MAX_LENGTH,
            do_sample=False,
            # num_beams=4,
            # early_stopping=True,
            # top_k=30, 
            # top_p=0.95
        )
        
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_predictions.extend([postprocess_translation(d).strip() for d in decoded])

pred = pd.DataFrame(
    {
        "predictions": all_predictions,
        "actuals": actuals
    }
    
)
pred

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


Starting Inference...


100%|██████████| 1/1 [00:00<00:00,  1.55it/s]


,predictions,actuals
0,To Ali-ahum from Damiq-pī-Aššur and also Puzur...,To Adida and Ali-ahum from Damiq-pī-Aššur: Our...
1,To Ali-ahum from Puzur-Aššur: Kulumaya should ...,To Ali-ahum from Zukuwa: Before I came your ti...
2,10,I left 62 kutānus with Ir'ibum and Šēp-Ištar
3,"To Ali-ahum, Lulu and Ennam-Aššur","To Adida, Ali-ahum and Kukkulānum from Puzur-A..."
4,"10 shekels of silver I gave to Iddin-Suen, son...","A box of Ili-bani, son of Mutalum, 3 shekels o..."
